# RSNA 2018 — Pipeline de Modelagem, Avaliação e Detecção

Este notebook executa o fluxo experimental pesado do baseline clássico: **validação das features, classificação, ablação, avaliação e detecção/localização**.
Os dados derivados ficam no Google Drive; o GitHub contém o código oficial e reproduzível.

In [1]:
BRANCH = "Erik"
!rm -rf ap1-rsna-pneumonia-baseline
!git clone -q -b {BRANCH} https://github.com/erik-hp/ap1-rsna-pneumonia-baseline.git
%cd ap1-rsna-pneumonia-baseline
!pip -q install -r requirements.txt
!pip -q install -e .
!pytest -q

/content/ap1-rsna-pneumonia-baseline
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.3/342.3 kB 24.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflic

## 1. Montar o Drive

A pasta esperada é `MyDrive/RSNA_TP1/data/processed/`.

In [2]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/RSNA_TP1")
PROCESSED = BASE / "data" / "processed"
OUT = BASE / "modelagem_avaliacao"
OUT.mkdir(parents=True, exist_ok=True)
FEATURES_ALL = PROCESSED / "features_all.csv"
FEATURES_REGIONS = PROCESSED / "features_regions.csv"
GROUND_TRUTH = PROCESSED / "ground_truth_boxes.csv"
IMAGE_MANIFEST = PROCESSED / "image_manifest.csv"
for p in [FEATURES_ALL, FEATURES_REGIONS, GROUND_TRUTH, IMAGE_MANIFEST]:
    print(("OK      " if p.exists() else "FALTANDO "), p)

Mounted at /content/drive
OK       /content/drive/MyDrive/RSNA_TP1/data/processed/features_all.csv
OK       /content/drive/MyDrive/RSNA_TP1/data/processed/features_regions.csv
OK       /content/drive/MyDrive/RSNA_TP1/data/processed/ground_truth_boxes.csv
OK       /content/drive/MyDrive/RSNA_TP1/data/processed/image_manifest.csv


## 2. Validar a matriz de classificação

In [3]:
!python scripts/validate_features.py --features "{FEATURES_ALL}" --auto-ablation

{
  "rows": 500,
  "unique_groups": 500,
  "classes": {
    "0": 387,
    "1": 113
  },
  "groups_per_class": {
    "0": 387,
    "1": 113
  },
  "positive_prevalence": 0.226,
  "feature_count_total": 346,
  "feature_counts_by_family": {
    "HOG": 324,
    "LBP": 10,
    "GLCM": 12,
    "HOG+LBP": 334,
    "HOG+GLCM": 336,
    "LBP+GLCM": 22,
    "HOG+LBP+GLCM": 346
  },
  "duplicate_full_rows": 0,
  "rows_beyond_one_per_group": 0,
  "has_candidate_boxes": false
}


## 3. Experimento principal — classificação por radiografia

Roda Dummy, SVM linear/RBF, Random Forest e HistGradientBoosting com nested CV 5×3 e todas as combinações HOG/LBP/GLCM.

In [4]:
CLASS_RESULTS = OUT / "classification" / "results"
CLASS_FIGURES = OUT / "classification" / "figures"
!python scripts/run_experiment.py --features "{FEATURES_ALL}" --auto-ablation --outer-splits 5 --inner-splits 3 --seed 42 --output-dir "{CLASS_RESULTS}" --figures-dir "{CLASS_FIGURES}" --plots best


Experimento concluído sem usar o teste oficial do Kaggle.
Resultados: /content/drive/MyDrive/RSNA_TP1/modelagem_avaliacao/classification/results
Figuras: /content/drive/MyDrive/RSNA_TP1/modelagem_avaliacao/classification/figures

Auditoria de entrada:
{
  "rows": 500,
  "unique_groups": 500,
  "classes": {
    "0": 387,
    "1": 113
  },
  "groups_per_class": {
    "0": 387,
    "1": 113
  },
  "positive_prevalence": 0.226,
  "feature_count_total": 346,
  "feature_counts_by_family": {
    "HOG": 324,
    "LBP": 10,
    "GLCM": 12,
    "HOG+LBP": 334,
    "HOG+GLCM": 336,
    "LBP+GLCM": 22,
    "HOG+LBP+GLCM": 346
  },
  "duplicate_full_rows": 0,
  "rows_beyond_one_per_group": 0,
  "has_candidate_boxes": false
}

Resumo (média ± desvio-padrão):
feature_family                  model       auc_roc        auc_pr   sensitivity   specificity            f1 balanced_accuracy
          GLCM                  dummy 0.500 ± 0.000 0.226 ± 0.049 0.000 ± 0.000 1.000 ± 0.000 0.000 ± 0.000     0.500 

## 4. Sensibilidade ao tratamento de desbalanceamento (opcional, mas já preparado)

Execute se houver tempo para comparar o estudo principal com os mesmos modelos sem `class_weight`.

In [5]:
NO_WEIGHT_RESULTS = OUT / "classification_no_class_weight" / "results"
NO_WEIGHT_FIGURES = OUT / "classification_no_class_weight" / "figures"
# Descomente para executar:
# !python scripts/run_experiment.py --features "{FEATURES_ALL}" --auto-ablation --class-weight none --outer-splits 5 --inner-splits 3 --seed 42 --output-dir "{NO_WEIGHT_RESULTS}" --figures-dir "{NO_WEIGHT_FIGURES}" --plots none

## 5. Detecção/localização — etapa pesada

Usa regiões candidatas, separação OOF por paciente, undersampling apenas no treino, HistGradientBoosting e NMS.

**Use o `features_regions.csv` regenerado com as 175 regiões da versão atual.**

In [6]:
DETECTION_RESULTS = OUT / "detection"
!python scripts/run_detection_experiment.py --features-regions "{FEATURES_REGIONS}" --ground-truth "{GROUND_TRUTH}" --image-manifest "{IMAGE_MANIFEST}" --model hist_gradient_boosting --family "HOG+LBP+GLCM" --outer-splits 5 --inner-splits 3 --max-neg-pos-ratio 3 --nms-iou 0.30 --top-k 20 --seed 42 --output-dir "{DETECTION_RESULTS}"

Fold 1/5: train=4,648 regiões, test=17,500, AUC-PR regional=0.1129
Fold 2/5: train=4,528 regiões, test=17,500, AUC-PR regional=0.1509
Fold 3/5: train=4,776 regiões, test=17,500, AUC-PR regional=0.1405
Fold 4/5: train=4,868 regiões, test=17,500, AUC-PR regional=0.1548
Fold 5/5: train=4,684 regiões, test=17,500, AUC-PR regional=0.1326
Exception ignored in: <function ResourceTracker.__del__ at 0x7b26265999e0>
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/usr/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/usr/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7aebdf19d9e0>
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/usr/lib/python3.12/multiprocessing/reso

## 6. Conferir os resultados para o artigo

In [ ]:
import json, pandas as pd
print("\n=== CLASSIFICAÇÃO ===")
display(pd.read_csv(CLASS_RESULTS / "summary_formatted.csv"))
print("\n=== DETECÇÃO ===")
print(json.dumps(json.loads((DETECTION_RESULTS / "detection_metrics.json").read_text()), indent=2))
print("\nArquivos finais em:", OUT)

## Saídas preservadas no Drive

- `modelagem_avaliacao/classification/results/`: métricas, ablação, parâmetros e erros;
- `modelagem_avaliacao/classification/figures/`: ROC, PR e matriz de confusão;
- `modelagem_avaliacao/detection/`: OOF regional, predições após NMS, mAP e FROC.

Não use resultados de smoke test no artigo.